<a href="https://colab.research.google.com/github/SarthoPramanik1075/Machine-Learning-/blob/main/Guassin_Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set(style='whitegrid')
plt.rcParams['figure.figsize']=(7,4)

In [ ]:
data=load_breast_cancer()
X=data.data
y=data.target

print('Shape of X (features):', X.shape)
print('Shape of y (labels):', y.shape)
print('Target names:', data.target_names)
print('\nFirst 5 feature names:')
print(data.feature_names[:5])

In [ ]:
# Put into a DataFrame for easier viewing
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
df.head()

In [ ]:
class_counts = df['target'].value_counts()
print(class_counts)

plt.figure()
class_counts.plot(kind='bar')
plt.xticks(ticks=[0,1], labels=data.target_names, rotation=0)
plt.title('Class distribution (0 = malignant, 1 = benign)')
plt.ylabel('Count')
plt.show()

In [ ]:
feature_name = 'mean radius'
feat_idx = list(data.feature_names).index(feature_name)

plt.figure()
plt.hist(X[y==0][:, feat_idx], bins=30, alpha=0.6, label='malignant')
plt.hist(X[y==1][:, feat_idx], bins=30, alpha=0.6, label='benign')
plt.legend()
plt.xlabel(feature_name)
plt.ylabel('Count')
plt.title(f'Distribution of {feature_name} by class')
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('Shape of X_train:', X_train.shape)
print('Shape of y_train:', y_train.shape)
print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)


##GuassinNB

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy on test set: ", acc)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion matrix:\n', cm)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names,
            yticklabels=data.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('GaussianNB – Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=data.target_names))

##MultinomialNB

In [ ]:
texts = [
    'I love this product',
    'This is amazing and fantastic',
    'I really like this',
    'I hate this item',
    'This is the worst thing ever',
    'Horrible and terrible experience',
]
labels = [1, 1, 1, 0, 0, 0]  # 1 = positive, 0 = negative

toy_df = pd.DataFrame({'text': texts, 'label': labels})
toy_df

In [ ]:
#convert text to feature with countvictorizer
vectorizer = CountVectorizer()
X_toy = vectorizer.fit_transform(toy_df['text'])
y_toy = toy_df['label']

print("Feature matrix shape: ", X_toy.shape)
print("Vocubulary: ", vectorizer.get_feature_names_out())


In [ ]:
mnb_toy = MultinomialNB()
mnb_toy.fit(X_toy, y_toy)

y_toy_pred = mnb_toy.predict(X_toy)
print('Accuracy on tiny dataset: ', accuracy_score(y_toy,y_toy_pred))
print('Classification repot: ')
print(classification_report(y_toy,y_toy_pred))

In [ ]:
new_texts = [
    'I love it',
    'This product is horrible',
    'Fantastic experience',
    'Worst purchase ever'
]
X_new = vectorizer.transform(new_texts)
new_pred = mnb_toy.predict(X_new)

for txt, pred in zip(new_texts, new_pred):
    label_str = 'positive' if pred == 1 else 'negative'
    print(f'{txt!r} -> {label_str}')

In [ ]:
!rm -rf ~/scikit_learn_data

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ['comp.graphics', 'rec.sport.baseball', 'sci.med']

newsgroups = fetch_20newsgroups(
    subset = 'train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    shuffle=True,
    random_state=42
)

In [ ]:
df_news = pd.DataFrame({
    'text': newsgroups.data,
    'label': newsgroups.target
})
df_news.head()

In [ ]:
print('Number of documents:', len(newsgroups.data))
print('Target names:', newsgroups.target_names)

In [ ]:
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    df_news['text'], df_news['label'], test_size=0.25, random_state=42
)

print('Train size:', X_train_text.shape[0])
print('Test size:', X_test_text.shape[0])

In [ ]:
vectorizer_news = CountVectorizer(stop_words='english', max_features=3000)
X_train_counts = vectorizer_news.fit_transform(X_train_text)
X_test_counts = vectorizer_news.transform(X_test_text)

print('Shape of X_train_counts:', X_train_counts.shape)
print('Shape of X_test_counts:', X_test_counts.shape)

In [ ]:
mnb_news = MultinomialNB()
mnb_news.fit(X_train_counts, y_train_text)
print('MultinomialNB model trained on news data.')

In [ ]:
y_news_pred = mnb_news.predict(X_test_counts)
acc_news = accuracy_score(y_test_text, y_news_pred)
print('Accuracy on 20 Newsgroups subset:', acc_news)

cm_news = confusion_matrix(y_test_text, y_news_pred)
print('Confusion matrix:\n', cm_news)

sns.heatmap(cm_news, annot=True, fmt='d', cmap='Greens',
            xticklabels=newsgroups.target_names,
            yticklabels=newsgroups.target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('MultinomialNB – 20 Newsgroups Subset')
plt.show()

print('Classification report:')
print(classification_report(y_test_text, y_news_pred,
                            target_names=newsgroups.target_names))